# The Wideband FMM Solver: Tree Structure and Algorithm

This notebook demonstrates the **FMMSolver**, **QuadTree**, and **FMMNode** classes
from `stablefmmpy`.

**Algorithm reference:**
- [HK] §4 Algorithm 4.1 — wideband FMM, two-regime (LF + HF)
- [M2D] §4 Algorithm 4.1 — single-regime FMM (non-oscillating kernels)

The FMM reduces the cost of computing $\phi = Kq$ from $O(N^2)$ to $O(N \cdot r)$
by grouping interactions hierarchically via a quad-tree.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stablefmmpy import (
    PointSet, HelmholtzKernel,
    FMMSolver, QuadTree, FMMNode,
)

print("stablefmmpy loaded.")
print(f"numpy {np.__version__}")


## How the FMM Works

The FMM partitions the point domain into a **quad-tree**:

1. **Tree construction:** Starting from a root box, nodes are recursively split
   into 4 quadrant children until each leaf holds ≤ N₀ points.
2. **Well-separated pairs:** Two leaves A and B are *well-separated* if
   $\delta_A + \delta_B \le \tau\,|o_A - o_B|$ (separation ratio $\tau$).
3. **M2L pass:** For well-separated leaf pairs, use the low-rank approximation
   $K \approx U_A\, B_{AB}\, V_B^T$ (interaction list, far field).
4. **P2P pass:** For nearby leaf pairs, compute the kernel directly (near list).

The **two-regime** (wideband) variant selects the LF or HF basis per leaf:
- **LF** ($k\delta \le r/e$): Bessel/Hankel recurrences with $\lambda$ balancing
- **HF** ($k\delta > r/e$): equispaced DFT basis (always numerically stable)


In [ ]:
# Point set configuration
rng  = np.random.default_rng(42)
N    = 200          # points per cluster
k    = 10.0         # wavenumber (LF regime: k*delta = 0.1 << r/e)

X = PointSet.random_uniform(N, center=0+0j,   radius=0.01, rng=rng)
Y = PointSet.random_uniform(N, center=0.1+0j, radius=0.01, rng=rng)
q = rng.standard_normal(N) + 1j * rng.standard_normal(N)

print(f"Sources Y: center={Y.center:.4g}, radius={Y.radius:.4g}")
print(f"Targets X: center={X.center:.4g}, radius={X.radius:.4g}")
print(f"Separation: d={abs(X.center - Y.center):.4g}")
tau = (X.radius + Y.radius) / abs(X.center - Y.center)
print(f"Separation ratio: tau = {tau:.3f}  (well-separated if tau < 0.6)")
print(f"LF regime check: k*delta = {k * X.radius:.4g}  (LF if <= r/e)")


## Section 1 — Quad-Tree Construction

The **QuadTree** partitions source and target points into a hierarchical set of boxes.
Each `FMMNode` stores: geometry (`center`, `radius`), point indices (`src_idx`, `tgt_idx`),
tree links (`parent`, `children`), and interaction/near lists.


In [ ]:
# Build the quad-tree (sources = Y, targets = X)
tree = QuadTree()
tree.build(Y.points, X.points, tau=0.6, N0=32)

# Tree statistics
all_nodes = tree._nodes
leaves    = [n for n in all_nodes if n.is_leaf()]
levels    = tree.by_level()

print(f"Total nodes : {len(all_nodes)}")
print(f"Leaf nodes  : {len(leaves)}")
print(f"Max level   : {tree.max_level()}")
print()
print(f"{'Level':>6}  {'Nodes':>6}")
for lv in sorted(levels):
    print(f"{lv:>6}  {len(levels[lv]):>6}")


In [ ]:
# Inspect FMMNode attributes on a sample source leaf and a sample target leaf.
# X and Y are well-separated (tau ~ 0.1), so the quad-tree never puts both
# a source point and a target point in the same leaf -- each leaf holds
# points from only one of the two clouds. Requiring both src_idx and tgt_idx
# to be non-empty (the original `and` check) matches no leaf and raises
# StopIteration.
src_leaf = next(n for n in leaves if len(n.src_idx) > 0)
tgt_leaf = next(n for n in leaves if len(n.tgt_idx) > 0)

print("Sample FMMNode (source leaf, holds Y points):")
print(f"  box_id          = {src_leaf.box_id}")
print(f"  level           = {src_leaf.level}")
print(f"  center          = {src_leaf.center:.6g}")
print(f"  radius          = {src_leaf.radius:.6g}")
print(f"  src_idx.shape   = {src_leaf.src_idx.shape}  (local indices into Y)")
print(f"  tgt_idx.shape   = {src_leaf.tgt_idx.shape}  (local indices into X)")
print(f"  interaction_list: {len(src_leaf.interaction_list)} far-field partners (M2L)")
print(f"  near_list       : {len(src_leaf.near_list)} near-field partners (P2P)")
print(f"  is_leaf()       = {src_leaf.is_leaf()}")
print()

print("Sample FMMNode (target leaf, holds X points):")
print(f"  box_id          = {tgt_leaf.box_id}")
print(f"  level           = {tgt_leaf.level}")
print(f"  center          = {tgt_leaf.center:.6g}")
print(f"  radius          = {tgt_leaf.radius:.6g}")
print(f"  src_idx.shape   = {tgt_leaf.src_idx.shape}  (local indices into Y)")
print(f"  tgt_idx.shape   = {tgt_leaf.tgt_idx.shape}  (local indices into X)")
print(f"  interaction_list: {len(tgt_leaf.interaction_list)} far-field partners (M2L)")
print(f"  near_list       : {len(tgt_leaf.near_list)} near-field partners (P2P)")
print(f"  is_leaf()       = {tgt_leaf.is_leaf()}")
print()

# Postorder vs preorder traversal
post = tree.postorder()
pre  = tree.preorder()
print(f"postorder() returns {len(post)} nodes (bottom-up, children before parent)")
print(f"preorder()  returns {len(pre)} nodes (top-down, parent before children)")


In [ ]:
# Visualize the quad-tree (boxes colored by level)
fig = tree.visualize(Y.points, X.points)
plt.savefig('02_quadtree.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_quadtree.png")


## Section 2 — FMM Solve

The **FMMSolver** implements the full wideband FMM algorithm:
- `solve(X, Y, q)` builds the tree internally and calls `_solve_wideband()`
- Per leaf, it selects LF or HF basis using the $k\delta$ criterion
- Result: $\phi_{\text{FMM}} \approx Kq$ in $O(N \cdot r)$ operations

Exact reference: `HelmholtzKernel.matvec()` computes $\phi = Kq$ in $O(N^2)$.


In [ ]:
r = 20    # expansion order

solver    = FMMSolver(k=k, r=r, tau=0.6, N0=32, balanced=True)
phi_fmm   = solver.solve(X, Y, q)

kern      = HelmholtzKernel(k)
phi_exact = kern.matvec(X, Y, q)

err = kern.relative_error(phi_fmm, phi_exact)
print(f"FMMSolver result (k={k}, r={r}, N={N}):")
print(f"  Relative error = {err:.3e}")
print(f"  phi_fmm  shape = {phi_fmm.shape}")
print(f"  phi_exact shape= {phi_exact.shape}")


## Section 3 — Accuracy vs. Expansion Order $r$

As $r$ increases, the low-rank approximation captures more terms in the
multipole expansion, and the error decreases exponentially until it reaches
machine precision.


In [ ]:
kern = HelmholtzKernel(k)
phi_exact = kern.matvec(X, Y, q)

r_sweep = [5, 10, 15, 20, 25, 30]
errs    = []

print(f"{'r':>4}  {'Relative error':>18}")
print("-" * 26)
for r_val in r_sweep:
    solver_r = FMMSolver(k=k, r=r_val, tau=0.6, N0=32, balanced=True)
    phi_r    = solver_r.solve(X, Y, q)
    e        = kern.relative_error(phi_r, phi_exact)
    errs.append(e)
    print(f"{r_val:>4}  {e:>18.6e}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(r_sweep, errs, 'b-o', lw=2, ms=7, label='Balanced FMM')
ax.axhline(1e-14, ls='--', color='gray', alpha=0.6, label='Machine precision (~1e-14)')
ax.set_xlabel('Expansion order $r$', fontsize=12)
ax.set_ylabel('Relative error', fontsize=12)
ax.set_title(f'FMM Accuracy vs. Expansion Order (k={k}, N={N})', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('02_fmm_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_fmm_accuracy.png")


## Section 4 — Interaction and Near Lists

Each leaf node carries two pre-computed lists used during the FMM solve:

| List | Usage | Criterion |
|------|-------|-----------|
| `interaction_list` | M2L (far-field, low-rank) | $\delta_A + \delta_B \le \tau|o_A - o_B|$ |
| `near_list` | P2P (near-field, direct) | not well-separated |

The near-field P2P cost is $O(N_0^2)$ per leaf pair; the M2L cost is $O(N_0 \cdot r)$.


In [ ]:
# Distribution of interaction_list and near_list sizes across all leaves
il_sizes = [len(n.interaction_list) for n in leaves]
nl_sizes = [len(n.near_list)        for n in leaves]

print(f"Interaction list (M2L, far-field):")
print(f"  min={min(il_sizes)}, max={max(il_sizes)}, mean={np.mean(il_sizes):.1f}")
print()
print(f"Near list (P2P, near-field):")
print(f"  min={min(nl_sizes)}, max={max(nl_sizes)}, mean={np.mean(nl_sizes):.1f}")
print()

# Well-separation check for a leaf pair
leaf_a = leaves[0]
leaf_b = leaves[1] if len(leaves) > 1 else leaves[0]
print(f"Example pair well-separated (tau=0.6): "
      f"{leaf_a.is_well_separated(leaf_b, tau=0.6)}")


## Conclusions

- The **QuadTree** adaptively partitions $N$ points into $O(N/N_0)$ leaves
  in $O(N \log N)$ time.
- **FMMSolver** computes $\phi \approx Kq$ with relative error $< 10^{-6}$
  for $r \ge 20$ in the low-frequency regime ($k=10$, $N=200$).
- Error decreases **exponentially** with $r$: each additional expansion order
  adds roughly one decimal digit of accuracy.
- The wideband solver selects the appropriate regime (LF or HF) per leaf,
  guaranteeing stability across all wavenumber scales.
